# 1.路由模式

```mermaid
flowchart LR
    A([query])
    B[Router]
    C[Agent A]
    D[Agent B]
    E[Agent C]
    F[Synthesize]
    G([Combined answer])
    A --> B
    B --> C
    B --> D
    B --> E
    C --> F
    D --> F
    E --> F
    F --> G

    style A fill:#00FFEE55, stroke:#008800,stroke-width:2px,color:#000000
    style G fill:#00FFEE55, stroke:#008800,stroke-width:2px,color:#000000
```

- command
    - 节点名称: 节点名称列表
    - Send -带参数的节点
- 问题分为三个：
    - GitHub
    - Notion
    - Slack

## 1.1 定义状态

In [27]:
from typing import Annotated,Literal,TypedDict
import operator

class AgentInput(TypedDict):
    """每个子agent的简单输入状态"""
    query:str

class AgentOutput(TypedDict):
    """每个子代理的输出"""
    source:str
    result:str

class Classification(TypedDict):
    """一个路由决策:使用哪个查询 调用什么代理"""
    source: Literal["github","notion","slack"]
    query:str

class RouterState(TypedDict):
    query :str
    classification:list[Classification]
    results:Annotated[list[AgentOutput],operator.add]
    final_answer:str

## 1.2 为每个垂直代理定义工具

- github : 搜索代码,获取主题，获取代码
- notion : 搜索文档,获取页面
- slack  : 搜索消息，获取线程

In [29]:
from langchain.tools import tool
#-------------------GitHub----------------------
@tool
def search_code(query:str,repo:str = "main") -> str:
    """在 GitHub 仓库中搜索代码"""
    return f"在 {repo} 仓库中找到与 '{query}' 匹配的代码,位于 src/auth.py 的身份验证中间件"

@tool
def search_issues(query:str) -> str:
    """搜索 GitHub 的议题拉去请求"""
    return f"找到3个与 '{query}' 匹配的议题:#142 (API认证文档)、·#89(OAuth流程)"

@tool
def search_prs(query:str) -> str:
    """搜索 GitHub 的search_prs"""
    return f"PR #156 添加了JWT认证"

#------------------notion------------------------
@tool
def search_notion(query:str) -> str :
    """在 Notion 工作区中搜索文档。"""
    return f"找到文档:《API 认证指南》——涵盖了 OAuth2 流程、API 密钥和 JWT 令牌"

@tool 
def get_page(page_id:str) ->str:
    """根据 ID 获取特定的 Notion 页面。"""
    return f"页面内容：分步身份验证设置说明"

#-------------------slack------------------------
@tool
def search_slack(query:str) -> str:
    """搜索 Slack 消息和线程。"""
    return f"在 #engineering 频道找到讨论：‘使用 Bearer 令牌进行 API 认证，刷新流程请参阅文档’"

@tool
def get_thread(thread_id:str)->str:
    """搜索 Slack 消息和线程。"""
    return f"在 #engineering 频道找到讨论：‘使用 Bearer 令牌进行 API 认证，刷新流程请参阅文档’"

## 1.3 创建专门化的代理

In [30]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="ollama:gemma4:31b",
    base_url="http://192.168.8.21:11434"
)

github_agent = create_agent(
    model=model,
    tools=[search_code,search_issues,search_prs],
    system_prompt=(
        "你是一位 GitHub 专家。通过搜索代码仓库、议题和拉取请求，回答关于代码、API 参考和实现细节的问题。"
    )
)

notion_agent = create_agent(
    model=model,
    tools=[search_notion,get_page],
    system_prompt=(
        "你是一位 Notion 专家。通过搜索组织的 Notion 工作区，回答关于内部流程、政策和团队文档的问题。"
    )
)

slack_agent = create_agent(
    model=model,
    tools=[search_slack,get_thread],
    system_prompt=(
        "你是一位 Slack 专家。通过搜索团队成员分享知识和解决方案的相关线程和讨论来回答问题。"
    )
)

## 1.4 构建路由工作流

- 构建AgentStates,继承BaseModel:存放分类结果
- 节点：
    - 分类查询
    - 条件分支：
        - GitHub 代理调用节点
        - notion 代理调用节点
        - slack  代理节点
    - 合并节点

In [31]:
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send    #任务分发

### 1. 定义状态（存放分类结果）

In [32]:
class ClassificationResult(BaseModel):
    """将用户的查询分类为特定的代理子问题的结果"""
    classification : list[Classification] = Field(description="要调用的代理列表及其对应的目标子问题。")

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

router_llm = init_chat_model(model="ollama:gemma4:e4b",base_url="http://192.168.8.21:11434")


### 2. 定义分类节点

In [33]:
from langchain.messages import SystemMessage,HumanMessage
def classify_query(state :RouterState) -> dict:
    """对查询分类，并且确定应该调用哪些代理"""

    #定义结构化输出的模型
    structured_llm = router_llm.with_structured_output(ClassificationResult)

    #调用
    system_prompt = """分析此查询，并确定需要查阅哪些知识库。
            对于每个相关的来源，生成一个针对该来源优化的、有目标的子问题。
            
            可用来源：
                - github：代码、API参考、实现细节、议题、拉取请求
                - notion：内部文档、流程、政策、团队维基
                - slack：团队讨论、非正式知识分享、最近的对话
                
            仅返回与查询相关的来源。每个来源应有一个针对其特定知识领域优化的、有目标的子问题。
            
            例如，对于"如何对API请求进行身份验证？"：
                - github："存在哪些身份验证代码？搜索认证中间件、JWT处理"
                - notion："存在哪些身份验证文档？查找API认证指南"
                （省略slack，因为它与此技术问题无关）"""
    
    user_prompt = state["query"]
    result = structured_llm.invoke([SystemMessage(system_prompt),HumanMessage(user_prompt)])

    return {
        "classification": result.classification
    }

### 3. 分发节点

In [34]:
def route_to_agent(state: RouterState) -> list[Send]:
    """根据分类结果 将任务分发给各个代理"""
    return [Send(node=c["source"],arg={"query":c["query"]}) for c in state['classification']]

### 4. 三个分支节点

In [35]:
# github节点 ：add_node的名字是GitHub
def query_github(state:AgentInput) -> dict:
    """调用 GitHub代理"""
    result = github_agent.invoke(
        {
            "messages":[
                {
                    "role":"user",
                    "content":state["query"]
                },
            ]
        }
    )
    #处理结果，返回给主路由
    return {
        "result":[
            {
                "source":"github",
                "result":result["messages"][-1].content
            }
        ]
        }

    

# notion节点： 
def query_notion(state:AgentInput) -> dict:
    result = notion_agent.invoke({
        "messages":[{"role":"user","content":state["query"]}]
    })
    
    return {
        "result":[
            {
                "source":"notion",
                "result":result["messages"][-1].content
            }
        ]
    }



# slack节点：
def query_slack(state:AgentInput) -> dict:
    result = slack_agent.invoke({
        "messages":[{"role":"user","content":state["query"]},]
    })

    return {
        "result":[
            {
                "source":"slack",
                "result":result["messages"][-1].content
            }
        ]
    }

### 5. 合并节点

In [36]:
def synthesis_results(state: RouterState) -> dict:
    """将所有代理的结果 组合为一个连贯的答案。"""

    # 判定三个代理的结果
    if not state['results']:
        return {
            "final_answer":"在知识源中，没有找到结果"
        }
    
    #将结果合成格式化的内容：提示词
    formatted = [
        f"**来自 {r['source']}**\n {r['result']}"
        for r in state["results"]
    ]

    #使用大模型润色
    synthesis_response = router_llm.invoke(
        [
            SystemMessage(f"综合这些搜索结果，来回答原始问题:{state['query']}"),    #系统提示词
            HumanMessage("\n".join(formatted))                                 #用户提示词
        ]
    )

    return{
        "final_answer":synthesis_response.content
    }

## 1.5 编译工作流

In [37]:
workflow = (
    #避免换行歧义
    StateGraph(RouterState)
        .add_node("classify",classify_query)
        .add_node("github",query_github)
        .add_node("notion",query_notion)
        .add_node("slack",query_slack)
        .add_node("synthesis",synthesis_results)

        .add_edge(START,"classify")
        .add_conditional_edges("classify", route_to_agent,["github","notion","slack"])
        .add_edge("github","synthesis")
        .add_edge("notion","synthesis")
        .add_edge("slack","synthesis")
        .add_edge("synthesis",END)
        .compile()
)

## 1.5 调用路由器

In [ ]:
result = workflow.invoke({
    "query": "我该如何验证 API 请求？"
})

print("原始查询：", result["query"])
print("\n分类结果:")
for c in result["classification"]:
    print(f"  {c['source']}: {c['query']}")
print("\n" + "=" * 60 + "\n")
print("最终答案：")
print(result["final_answer"])